<a href="https://colab.research.google.com/github/SebaSRomanH/NBA/blob/main/Data/NBA_FG3M_EDA_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Alcance inicial

Este archivo está dedicado exclusivamente al **análisis exploratorio de datos**. Incluye:

1. carga y revisión de estructura;
2. interpretación automática de la nomenclatura NBA;
3. normalización Min–Max de variables numéricas al rango `[0,1]`;
4. calidad, distribución y relaciones con `TARGET_FG3M`;
5. clustering jerárquico de variables;
6. gráfico interactivo de dispersión con línea de tendencia.

No entrena modelos predictivos ni exporta artefactos.


---
## FASE 1 · Contexto y alcance analítico

<div style="border-left:4px solid #10B0D0;padding:12px 18px;background:#0D1F2A;margin:12px 0;border-radius:0 4px 4px 0;font-family:Arial,sans-serif;">
<strong style="color:#10B0D0;">Contexto NBA — triples</strong><br>
<span style="font-size:13px;color:#A8B8C8;line-height:1.7;">
Un <strong style="color:#E8EAF0;">triple</strong> es un lanzamiento convertido desde fuera de la línea de tres puntos.
En la nomenclatura estadística NBA, <strong style="color:#E8EAF0;">FG3M</strong> representa triples convertidos,
<strong style="color:#E8EAF0;">FG3A</strong> triples intentados y <strong style="color:#E8EAF0;">FG3_PCT</strong> el porcentaje de acierto.
El análisis busca comprender la distribución de `TARGET_FG3M`, la calidad del dataset, la redundancia entre variables
y las asociaciones estadísticas con el rendimiento de triples.
</span>
</div>

### Objetivos del EDA

| Objetivo | Descripción |
|---|---|
| **Interpretación** | Traducir abreviaturas y ventanas temporales de las variables NBA. |
| **Calidad** | Detectar nulos, constantes, identificadores y posibles variables con información futura. |
| **Distribución** | Analizar forma, dispersión y valores extremos de `TARGET_FG3M`. |
| **Relaciones** | Explorar correlaciones y tendencias entre la variable dependiente y las explicativas. |
| **Estructura** | Agrupar variables similares mediante clustering jerárquico. |

### Consideraciones

- `NEXT_GAME_*` se identifica como posible **fuga de información temporal**.
- Las columnas `TARGET_*` distintas de `TARGET_FG3M` deben revisarse según el momento en que estarían disponibles.
- La normalización Min–Max se usa para facilitar comparaciones de escala; las variables originales se conservan en `df`.


---
## FASE 2 · Comprensión y análisis exploratorio de los datos

### 2.1 Configuración visual y operativa


In [ ]:
# @title
import warnings
warnings.filterwarnings("ignore")

import io, base64
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# ── Paleta vibrante sobre oscuro ──────────────────────────────────
C1 = "#F05030"   # rojo-coral
C2 = "#10B0D0"   # cyan
C3 = "#F0B010"   # ámbar
C4 = "#9070B0"   # violeta
C5 = "#30B0B0"   # teal
C6 = "#4090B0"   # azul-gris
C7 = "#F07030"   # naranja
C8 = "#607080"   # gris azulado

PALETTE = [C1, C2, C3, C4, C5, C6, C7, C8]

BG    = "#12151C"
BG2   = "#1A1E28"
BG3   = "#1E2330"
TEXT  = "#E8EAF0"
MUTED = "#556070"

CMAP_DIV = LinearSegmentedColormap.from_list("dark_div", [C2, BG3, C1])
CMAP_SEQ = LinearSegmentedColormap.from_list("dark_seq", [BG2, C1])

plt.rcParams.update({
    "figure.facecolor": BG,  "axes.facecolor": BG,
    "axes.edgecolor":   BG3, "axes.labelcolor": MUTED,
    "axes.titlecolor":  TEXT,"axes.titlesize":  12,
    "axes.titleweight": "bold","axes.titlepad":  14,
    "axes.labelsize":   9,   "axes.grid":       True,
    "grid.color":       BG3, "grid.linestyle":  "-",
    "grid.linewidth":   0.8, "grid.alpha":      1.0,
    "xtick.color":      MUTED,"ytick.color":    MUTED,
    "xtick.labelsize":  8,   "ytick.labelsize": 8,
    "text.color":       TEXT,"legend.facecolor": BG2,
    "legend.edgecolor": BG3, "legend.labelcolor": TEXT,
    "legend.fontsize":  8,   "axes.spines.top":  False,
    "axes.spines.right":False,"axes.spines.left": True,
    "axes.spines.bottom":True,"figure.dpi":       110,
    "figure.figsize":   (13, 4.5),
})

# ── UI helpers ────────────────────────────────────────────────────
def section(label, sub="", color=C1):
    sub_html = (f'<span style="font-size:11px;color:{MUTED};display:block;margin-top:3px;">{sub}</span>'
                if sub else "")
    display(HTML(
        f'<div style="border-left:4px solid {color};padding:8px 0 8px 16px;'
        f'margin:24px 0 12px;background:{BG2};border-radius:0 4px 4px 0;font-family:Arial,sans-serif;">'
        f'<span style="font-size:12px;font-weight:700;color:{TEXT};'
        f'text-transform:uppercase;letter-spacing:0.5px;">{label}</span>'
        f'{sub_html}</div>'
    ))

def metrics(stats: dict, color=C1):
    cards = "".join([
        f'<div style="padding:12px 20px;border-top:3px solid {color};'
        f'background:{BG2};min-width:105px;border-radius:0 0 4px 4px;">'
        f'<div style="font-size:9px;letter-spacing:1.5px;text-transform:uppercase;'
        f'color:{MUTED};font-family:Arial,sans-serif;margin-bottom:6px;">{k}</div>'
        f'<div style="font-size:22px;font-weight:700;color:{TEXT};font-family:Arial,sans-serif;">{v}</div>'
        f'</div>'
        for k, v in stats.items()
    ])
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:6px;margin:10px 0 18px;">{cards}</div>'))

def alert(msg, tipo="info"):
    cfg = {
        "info":    (C2, "#0D1F2A", "Nota"),
        "warning": (C3, "#1F1A08", "Atención"),
        "danger":  (C1, "#1F0D08", "Leakage"),
        "ok":      (C5, "#0A1A1A", ""),
        "context": (C4, "#150F20", "Contexto NBA"),
        "decision":(C3, "#1A1500", "Decisión metodológica"),
    }.get(tipo, (C2, "#0D1F2A", "Nota"))
    color, bg, prefix = cfg
    pfx = f'<strong style="color:{color};">{prefix}:</strong> ' if prefix else ""
    display(HTML(
        f'<div style="border-left:4px solid {color};background:{bg};'
        f'padding:10px 16px;margin:8px 0 14px;border-radius:0 4px 4px 0;'
        f'font-family:Arial,sans-serif;font-size:12px;color:{TEXT};line-height:1.7;">{pfx}{msg}</div>'
    ))

def tbl(df_styler):
    return df_styler.set_table_styles([
        {"selector":"thead th","props":[
            ("background",BG3),("color",MUTED),("font-size","9px"),
            ("letter-spacing","1.5px"),("text-transform","uppercase"),
            ("font-family","Arial,sans-serif"),("font-weight","700"),
            ("padding","9px 13px"),("border-bottom",f"2px solid {C1}"),
            ("text-align","left")]},
        {"selector":"tbody td","props":[
            ("background",BG),("color",TEXT),("font-size","11px"),
            ("font-family","monospace"),("padding","7px 13px"),
            ("border-bottom",f"1px solid {BG3}")]},
        {"selector":"tbody tr:hover td","props":[("background",BG2)]},
        {"selector":"table","props":[("border-collapse","collapse"),("width","100%")]},
    ])

def zoom_plot(fig, title="Gráfico"):
    """Botón para abrir el gráfico en pantalla completa (nueva pestaña)."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(HTML(
        f'<div style="margin:6px 0 14px;">'
        f'<a href="data:image/png;base64,{b64}" target="_blank" '
        f'style="display:inline-flex;align-items:center;gap:7px;'
        f'background:{BG2};border:1px solid {BG3};border-left:3px solid {C2};'
        f'border-radius:0 6px 6px 0;padding:6px 16px;font-family:Arial,sans-serif;'
        f'font-size:11px;color:{C2};text-decoration:none;letter-spacing:0.3px;font-weight:500;">'
        f'&#x26F6;&nbsp; Abrir en pantalla completa — {title}</a></div>'
    ))

print("✓ Configuración visual dark cargada.")


# Habilita widgets interactivos en Google Colab cuando corresponde.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


### 2.2 Configuración operativa


In [ ]:
# @title
# ════════════════════════════════════════════════════════
#  CONFIGURACIÓN CENTRAL — análisis exploratorio
# ════════════════════════════════════════════════════════

CONFIG = {
    # ── GitHub privado ──────────────────────────────────
    "GITHUB_OWNER":  "SebaSRomanH",
    "GITHUB_REPO":   "NBA",
    "GITHUB_BRANCH": "main",
    "GITHUB_PATH":   "Data/dataset_nba.xlsx",

    # ── Formato ─────────────────────────────────────────
    "SHEET_NAME": "Hoja 1",
    "DECIMAL":    ",",

    # ── Variable dependiente del EDA ────────────────────
    "TARGET_COL": "TARGET_FG3M",

    # ── Ventanas temporales esperadas ───────────────────
    "WINDOWS": ["SEASON", "L20", "L10", "L5", "L3"],

    # ── Clustering jerárquico ───────────────────────────
    "CLUSTER_N_GROUPS": 6,
    "CLUSTER_LINKAGE":  "ward",

    # ── Normalización ───────────────────────────────────
    "SCALING_MIN": 0.0,
    "SCALING_MAX": 1.0,

    # ── Gráfico interactivo ─────────────────────────────
    "SCATTER_MAX_POINTS": 4000,

    # ── Identificadores que no participan en análisis ──
    "ID_COLS": ["PLAYER_ID", "TARGET_GAME_ID", "LAST_GAME_ID",
                "TARGET_TEAM_ID", "PLAYER_NAME_RAW"],

    # ── Umbrales de calidad ─────────────────────────────
    "NULL_THRESHOLD": 0.30,
    "CARD_THRESHOLD": 50,
    "MAX_CATS":       15,
    "SEED":           42,
}

np.random.seed(CONFIG["SEED"])

section("Configuración operativa", "Parámetros centralizados del EDA", C2)
metrics({
    "Dependiente": CONFIG["TARGET_COL"],
    "Propósito":   "EDA",
    "Ventanas":    " · ".join(CONFIG["WINDOWS"]),
    "Clusters":    str(CONFIG["CLUSTER_N_GROUPS"]),
    "Escala":      "[0,1]",
}, C2)
alert(
    "El notebook conserva df con valores originales y crea df_scaled para la versión normalizada. "
    "No se realizan entrenamiento ni exportación de artefactos.", "decision"
)

### 2.3 Autenticación GitHub (repositorio privado)


In [ ]:
# @title
from getpass import getpass
import urllib.request, json as _j, base64 as _b64
from pathlib import Path
from io import BytesIO

USE_COLAB_SECRETS = False

section("Autenticación GitHub privado", "Personal Access Token — scope: repo", C4)

if USE_COLAB_SECRETS:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_PAT")
        alert("Token cargado desde Colab Secrets.", "ok")
    except Exception as e:
        alert(f"Secret GITHUB_PAT no encontrado: {e}", "danger")
        GITHUB_TOKEN = None
else:
    print("Ingresa tu GitHub Personal Access Token (scope: repo).")
    GITHUB_TOKEN = getpass("GitHub PAT: ")
    if GITHUB_TOKEN:
        alert("Token recibido. No queda almacenado en el notebook.", "ok")
    else:
        alert("Token vacío. Vuelve a ejecutar esta celda.", "danger")

def _verify_repo(owner, repo, token):
    req = urllib.request.Request(
        f"https://api.github.com/repos/{owner}/{repo}",
        headers={"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            d = _j.loads(r.read())
            alert(f"Repo accesible: {owner}/{repo} ({'privado' if d.get('private') else 'público'})", "ok")
            return True
    except urllib.error.HTTPError as e:
        alert({401: "Token inválido.", 404: f"Repo {owner}/{repo} no encontrado."
               }.get(e.code, f"HTTP {e.code}"), "danger")
        return False

if GITHUB_TOKEN:
    _verify_repo(CONFIG["GITHUB_OWNER"], CONFIG["GITHUB_REPO"], GITHUB_TOKEN)


### 2.4 Carga del dataset


In [ ]:
# @title
section("Carga del dataset", f"{CONFIG['GITHUB_OWNER']}/{CONFIG['GITHUB_REPO']} — {CONFIG['GITHUB_PATH']}", C1)

def fetch_github(owner, repo, path, branch, token):
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={branch}"
    print(f"Fetching from URL: {url}")
    req = urllib.request.Request(url, headers={
        "Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        meta = _j.loads(r.read())
    print(f"GitHub API metadata response: {meta}")

    # Prioritize download_url if available, as it directly provides the raw file.
    if "download_url" in meta and meta["download_url"]:
        print(f"Using download URL: {meta['download_url']}")
        req2 = urllib.request.Request(meta["download_url"],
                                       headers={"Authorization": f"token {token}"})
        with urllib.request.urlopen(req2, timeout=60) as r:
            return r.read()
    elif meta.get("encoding") == "base64":
        print("Using base64 content from metadata.")
        return _b64.b64decode(meta["content"].replace("\n", ""))
    else:
        raise ValueError(f"Could not get file content. Metadata: {meta}")

def parse_excel(raw, sheet, decimal=","):
    df = pd.read_excel(BytesIO(raw), sheet_name=sheet, engine='openpyxl')
    for col in df.select_dtypes("object").columns:
        conv = df[col].astype(str).str.replace(decimal, ".", regex=False).str.strip()
        try:
            df[col] = pd.to_numeric(conv)
        except Exception:
            pass
    return df

def normalize_cols(df):
    m = {c: c.strip().upper().replace(" ", "_") for c in df.columns}
    return df.rename(columns=m), m

try:
    _raw = fetch_github(CONFIG["GITHUB_OWNER"], CONFIG["GITHUB_REPO"],
                        CONFIG["GITHUB_PATH"], CONFIG["GITHUB_BRANCH"], GITHUB_TOKEN)
    alert(f"Archivo descargado — {len(_raw)/1024:.1f} KB", "ok")
    df_original = parse_excel(_raw, CONFIG["SHEET_NAME"], CONFIG["DECIMAL"])
    df_original, _ = normalize_cols(df_original)
    df = df_original.copy()
    metrics({
        "Filas":    f"{df.shape[0]:,}",
        "Columnas": f"{df.shape[1]:,}",
        "Memoria":  f"{df.memory_usage(deep=True).sum()/1e6:.1f} MB",
    }, C1)
    alert("df_original preservado sin modificaciones. Trabajo sobre copia df.", "info")
    display(tbl(df.head(3).style))
except Exception as e:
    print(f"Error during data loading or parsing: {e}")
    alert(f"Error en carga: {e}", "danger")

### 2.5 Capa de interpretación de variables

La nomenclatura combina una **función estadística**, una **métrica NBA** y una **ventana temporal**.

Ejemplo:


La variable `MIN_DREB_L10` se interpreta como:

- `MIN`: valor mínimo observado;
- `DREB`: rebotes defensivos;
- `L10`: últimos 10 partidos.

Por tanto, representa el **mínimo de rebotes defensivos registrado en los últimos 10 partidos**.


In [ ]:
# @title
section("Interpretación de variables", "Glosario NBA y descripción automática por columna", C4)

# Métricas y abreviaturas frecuentes en datasets NBA.
NBA_TERMS = {
    "FG3_PCT": "porcentaje de acierto en triples",
    "FG_PCT": "porcentaje de acierto en tiros de campo",
    "FT_PCT": "porcentaje de acierto en tiros libres",
    "OFF_RATING": "rating ofensivo",
    "DEF_RATING": "rating defensivo",
    "NET_RATING": "rating neto",
    "PLUS_MINUS": "diferencial de puntos mientras el jugador está en cancha",
    "DAYS_REST": "días de descanso",
    "HOME_AWAY": "condición de local o visitante",
    "FG3M": "triples convertidos",
    "FG3A": "triples intentados",
    "FGM": "tiros de campo convertidos",
    "FGA": "tiros de campo intentados",
    "FTM": "tiros libres convertidos",
    "FTA": "tiros libres intentados",
    "DREB": "rebotes defensivos",
    "OREB": "rebotes ofensivos",
    "REB": "rebotes totales",
    "AST": "asistencias",
    "STL": "robos de balón",
    "BLK": "tapones o bloqueos",
    "TOV": "pérdidas de balón",
    "PF": "faltas personales",
    "PTS": "puntos anotados",
    "MIN": "minutos jugados",
    "POSS": "posesiones estimadas",
    "PACE": "ritmo de juego",
    "USG_PCT": "porcentaje de uso ofensivo",
    "TS_PCT": "porcentaje de tiro verdadero",
    "EFG_PCT": "porcentaje efectivo de tiros de campo",
    "PIE": "Player Impact Estimate",
    "AGE": "edad del jugador",
    "HEIGHT": "altura del jugador",
    "WEIGHT": "peso del jugador",
    "EXPERIENCE": "años de experiencia",
}

STAT_PREFIXES = {
    "AVG": "promedio",
    "MEAN": "promedio",
    "MIN": "mínimo",
    "MAX": "máximo",
    "SUM": "suma",
    "STD": "desviación estándar",
    "VAR": "varianza",
    "MEDIAN": "mediana",
    "CV": "coeficiente de variación",
    "TREND": "tendencia",
    "LAST": "último valor",
}

WINDOW_TEXT = {
    "L3": "los últimos 3 partidos",
    "L5": "los últimos 5 partidos",
    "L10": "los últimos 10 partidos",
    "L20": "los últimos 20 partidos",
    "SEASON": "la temporada",
}


def _contains_term(name, term):
    padded = f"_{name}_"
    return f"_{term}_" in padded


def classify_col(col, target, id_cols):
    if col in id_cols or col.endswith("_ID"):       return "identificador"
    if col == target:                                return "variable_dependiente"
    if col.startswith("NEXT_GAME_"):                return "posible_leakage"
    if col.startswith("TARGET_") and col != target: return "contexto_objetivo_revisar"
    if "_L3" in col or col.endswith("_L3"):        return "ventana_L3"
    if "_L5" in col or col.endswith("_L5"):        return "ventana_L5"
    if "_L10" in col or col.endswith("_L10"):      return "ventana_L10"
    if "_L20" in col or col.endswith("_L20"):      return "ventana_L20"
    if "SEASON" in col:                             return "ventana_season"
    if "TREND" in col:                              return "tendencia"
    if "PER_MIN" in col or "PER_GAME" in col:      return "eficiencia"
    if col in ["AGE", "HEIGHT", "WEIGHT", "PLAYER_POSITION", "SCHOOL",
               "BIRTH_DATE", "PLAYER_NAME", "COUNTRY", "EXPERIENCE"]:
        return "perfil_jugador"
    return "otra"


def describe_variable(col):
    name = str(col).upper().strip()
    tokens = name.split("_")
    role = classify_col(name, CONFIG["TARGET_COL"], CONFIG["ID_COLS"])

    if name == CONFIG["TARGET_COL"]:
        return "Triples convertidos por el jugador en el partido objetivo; variable dependiente del análisis."
    if role == "identificador":
        return "Identificador o etiqueta de registro; no representa una magnitud deportiva continua."

    window_key = next((w for w in WINDOW_TEXT if _contains_term(name, w)), None)

    # MIN puede significar minutos o mínimo. Es prefijo estadístico cuando precede otra métrica.
    stat_key = None
    if tokens and tokens[0] in STAT_PREFIXES:
        if tokens[0] != "MIN" or (len(tokens) > 1 and tokens[1] not in WINDOW_TEXT):
            stat_key = tokens[0]

    searchable = name
    if stat_key:
        searchable = "_".join(tokens[1:])

    metric_key = next(
        (k for k in sorted(NBA_TERMS, key=len, reverse=True) if _contains_term(searchable, k)),
        None
    )

    # Caso MIN_L10: MIN es minutos, no mínimo.
    if metric_key is None and _contains_term(name, "MIN"):
        metric_key = "MIN"
        if stat_key == "MIN":
            stat_key = None

    metric_text = NBA_TERMS.get(metric_key)
    stat_text = STAT_PREFIXES.get(stat_key)

    if metric_text and stat_text:
        description = f"{stat_text.capitalize()} de {metric_text}"
    elif metric_text:
        description = metric_text.capitalize()
    else:
        core_tokens = [t for t in tokens if t not in WINDOW_TEXT and t not in STAT_PREFIXES
                       and t not in {"TARGET", "NEXT", "GAME"}]
        readable = " ".join(core_tokens).lower() or name.lower().replace("_", " ")
        description = f"Variable asociada a {readable}"

    if window_key:
        description += f" en {WINDOW_TEXT[window_key]}"
    if "PER_MIN" in name:
        description += ", expresada por minuto jugado"
    if "PER_GAME" in name:
        description += ", expresada por partido"
    if name.startswith("TARGET_") and name != CONFIG["TARGET_COL"]:
        description += ". Corresponde al partido objetivo y debe revisarse su disponibilidad temporal"
    if name.startswith("NEXT_GAME_"):
        description += ". Contiene información futura y se marca como posible fuga de información"

    return description.rstrip(".") + "."


def detected_terms(col):
    name = str(col).upper()
    terms = [k for k in sorted(NBA_TERMS, key=len, reverse=True) if _contains_term(name, k)]
    windows = [w for w in WINDOW_TEXT if _contains_term(name, w)]
    return ", ".join(terms + windows) if terms or windows else "—"

variable_dictionary = pd.DataFrame({
    "columna": df.columns,
    "grupo": [classify_col(c, CONFIG["TARGET_COL"], CONFIG["ID_COLS"]) for c in df.columns],
    "abreviaturas": [detected_terms(c) for c in df.columns],
    "descripcion_breve": [describe_variable(c) for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
    "nulos": [int(df[c].isna().sum()) for c in df.columns],
})

detected = sorted({
    term
    for col in df.columns
    for term in NBA_TERMS
    if _contains_term(str(col).upper(), term)
})

glossary_df = pd.DataFrame({
    "abreviatura": detected,
    "significado": [NBA_TERMS[t] for t in detected],
})

metrics({
    "Variables": str(len(variable_dictionary)),
    "Términos NBA": str(len(glossary_df)),
    "Ventanas": str(sum(variable_dictionary["abreviaturas"].str.contains("L3|L5|L10|L20|SEASON", regex=True))),
}, C4)

print("Glosario NBA detectado en el dataset")
display(tbl(glossary_df.style))

print("Diccionario de variables")
display(tbl(variable_dictionary.style))


### 2.6 Capa de estandarización Min–Max `[0,1]`

Se aplica la transformación:

\[
x' = \frac{x-\min(x)}{\max(x)-\min(x)}
\]

El resultado se guarda en `df_scaled`. `df` conserva los valores originales. Para las visualizaciones donde la variable dependiente debe seguir interpretándose como número de triples, se utiliza `TARGET_FG3M` en su escala original.


In [ ]:
# @title
section("Estandarización Min–Max", "Variables numéricas transformadas al rango [0,1]", C2)


def is_identifier_like(col):
    return col in CONFIG["ID_COLS"] or col.endswith("_ID") or col.endswith("GAME_ID")

numeric_cols_all = df.select_dtypes(include="number").columns.tolist()
scale_cols = [c for c in numeric_cols_all if not is_identifier_like(c)]

df_scaled = df.copy()
scaling_rows = []

for col in scale_cols:
    s = pd.to_numeric(df[col], errors="coerce")
    old_min = s.min(skipna=True)
    old_max = s.max(skipna=True)

    if pd.isna(old_min) or pd.isna(old_max):
        df_scaled[col] = s
        status = "sin datos válidos"
    elif old_max > old_min:
        df_scaled[col] = (
            CONFIG["SCALING_MIN"]
            + (s - old_min) * (CONFIG["SCALING_MAX"] - CONFIG["SCALING_MIN"]) / (old_max - old_min)
        )
        status = "normalizada"
    else:
        # Una constante no tiene rango; se representa en 0 para mantener [0,1].
        df_scaled[col] = np.where(s.notna(), CONFIG["SCALING_MIN"], np.nan)
        status = "constante → 0"

    scaling_rows.append({
        "columna": col,
        "min_original": old_min,
        "max_original": old_max,
        "min_scaled": pd.to_numeric(df_scaled[col], errors="coerce").min(skipna=True),
        "max_scaled": pd.to_numeric(df_scaled[col], errors="coerce").max(skipna=True),
        "estado": status,
    })

scaling_report = pd.DataFrame(scaling_rows)

# df_eda se usa cuando las explicativas deben estar normalizadas, pero el objetivo
# se mantiene en cantidad real de triples para conservar interpretación.
df_eda = df_scaled.copy()
if CONFIG["TARGET_COL"] in df.columns:
    df_eda[CONFIG["TARGET_COL"]] = df[CONFIG["TARGET_COL"]]

n_ok = (scaling_report["estado"] == "normalizada").sum()
n_const = scaling_report["estado"].str.contains("constante", na=False).sum()
metrics({
    "Numéricas": str(len(numeric_cols_all)),
    "Normalizadas": str(n_ok),
    "Constantes": str(n_const),
    "Rango": "0.000–1.000",
}, C2)

alert(
    "df_scaled contiene la versión Min–Max. df_eda conserva TARGET_FG3M en escala original "
    "y mantiene las variables explicativas normalizadas.", "ok"
)

display(tbl(
    scaling_report.style
    .format({"min_original": "{:.4f}", "max_original": "{:.4f}",
             "min_scaled": "{:.4f}", "max_scaled": "{:.4f}"})
))

preview_cols = scale_cols[:12]
if preview_cols:
    print("Vista previa de variables normalizadas")
    display(tbl(df_scaled[preview_cols].head(5).style.format("{:.3f}")))


### 2.7 Estructura y grupos de variables

Las variables se clasifican por su función analítica, su ventana temporal y su posible disponibilidad antes del partido objetivo.


In [ ]:
# @title
section("Estructura de variables", "Grupos temáticos y diccionario interpretado", C3)

col_meta = []
for col in df.columns:
    s = df[col]
    grupo = classify_col(col, CONFIG["TARGET_COL"], CONFIG["ID_COLS"])
    col_meta.append({
        "columna": col,
        "grupo": grupo,
        "descripcion_breve": describe_variable(col),
        "dtype": str(s.dtype),
        "unicos": s.nunique(),
        "nulos": s.isna().sum(),
    })

col_meta_df = pd.DataFrame(col_meta)
grupo_counts = col_meta_df["grupo"].value_counts()

metrics({g.replace("_", " "): str(v) for g, v in grupo_counts.items()}, C3)

fig, ax = plt.subplots(figsize=(10, max(4, len(grupo_counts) * 0.6)))
bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(grupo_counts))]
brs = ax.barh(grupo_counts.index[::-1], grupo_counts.values[::-1],
              color=bar_colors[::-1], height=0.6, zorder=3)
for bar, val in zip(brs, grupo_counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va="center", color=MUTED, fontsize=8)
ax.set_title("Variables por grupo temático", pad=14)
ax.set_xlabel("Cantidad de columnas", color=MUTED, fontsize=8)
ax.spines["left"].set_color(BG3)
ax.spines["bottom"].set_color(BG3)
plt.tight_layout()
zoom_plot(fig, "Grupos de variables")
plt.show()

display(tbl(col_meta_df.style))


### 2.8 Calidad de datos


In [ ]:
# @title
section("Calidad de datos", "Nulos · Constantes · Tipo · Temporalidad", C1)


def quality_report(data, null_thr=0.30):
    n = len(data)
    rows = []
    for col in data.columns:
        s = data[col]
        is_num = pd.api.types.is_numeric_dtype(s)
        nulos = s.isna().sum()
        pct_n = nulos / max(n, 1) * 100
        unicos = s.nunique()
        ceros = int((s == 0).sum()) if is_num else 0
        inf_v = int(np.isinf(pd.to_numeric(s, errors="coerce")).sum()) if is_num else 0
        warns = []
        if pct_n > null_thr * 100:                       warns.append(f"nulos>{null_thr*100:.0f}%")
        if unicos <= 1:                                  warns.append("constante")
        if unicos / max(n, 1) > 0.95 and unicos > 10:   warns.append("posible_id")
        if inf_v > 0:                                    warns.append(f"inf:{inf_v}")
        if col.startswith("NEXT_GAME_"):                warns.append("POSIBLE_LEAKAGE")
        if col.startswith("TARGET_") and col != CONFIG["TARGET_COL"]:
            warns.append("REVISAR_TEMPORALIDAD")
        rows.append({
            "columna": col,
            "dtype": str(s.dtype),
            "nulos": nulos,
            "pct_nulos": round(pct_n, 1),
            "ceros": ceros,
            "unicos": unicos,
            "pct_unicos": round(unicos / max(n, 1) * 100, 1),
            "status": " · ".join(warns) if warns else "ok",
        })
    return pd.DataFrame(rows)

quality_df = quality_report(df, CONFIG["NULL_THRESHOLD"])
n_warn = (quality_df["status"] != "ok").sum()
n_leak = quality_df["status"].str.contains("POSIBLE_LEAKAGE").sum()

metrics({
    "Con observaciones": str(n_warn),
    "Posible leakage": str(n_leak),
    "Sin alertas": str(len(quality_df) - n_warn),
}, C1)

_wc = quality_df["status"].str.split(" · ").explode()
_wc = _wc[_wc != "ok"].value_counts()
if len(_wc):
    fig, ax = plt.subplots(figsize=(9, max(3, len(_wc) * 0.6)))
    _bc = [C1 if "LEAKAGE" in w else C3 if "nulos" in w or "TEMPORALIDAD" in w else C2 for w in _wc.index]
    brs = ax.barh(_wc.index[::-1], _wc.values[::-1], color=_bc[::-1], height=0.5, zorder=3)
    for bar, val in zip(brs, _wc.values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                str(val), va="center", color=MUTED, fontsize=8)
    ax.set_title("Observaciones de calidad por categoría", pad=14)
    ax.set_xlabel("Columnas afectadas", color=MUTED, fontsize=8)
    ax.spines["left"].set_color(BG3)
    ax.spines["bottom"].set_color(BG3)
    plt.tight_layout()
    zoom_plot(fig, "Calidad de datos")
    plt.show()

if n_leak:
    alert(f"Se detectaron {n_leak} columnas NEXT_GAME_* con información potencialmente posterior al partido analizado.", "danger")


def _hl_status(val):
    if isinstance(val, str) and "LEAKAGE" in val: return f"color:{C1};font-weight:700"
    if isinstance(val, str) and val != "ok":      return f"color:{C3};font-weight:600"
    if val == "ok":                              return f"color:{C5}"
    return ""

display(tbl(
    quality_df.style
    .map(_hl_status, subset=["status"])
    .format({"pct_nulos": "{:.1f}%", "pct_unicos": "{:.1f}%"})
))


### 2.9 Estadísticas descriptivas y distribución de la variable dependiente

`TARGET_FG3M` es una variable discreta de conteo. La relación entre media y varianza se presenta como una característica descriptiva de su dispersión, sin entrenar ningún modelo.


In [ ]:
# @title
section("Estadísticas descriptivas", "Escala original · Distribución de TARGET_FG3M", C2)

num_cols = df.select_dtypes(include="number").columns.tolist()
desc = df[num_cols].describe().T
desc["skewness"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()

display(tbl(
    desc.style.format("{:.3f}")
    .background_gradient(cmap=CMAP_SEQ, subset=["mean", "std"])
))

target = CONFIG["TARGET_COL"]
if target in df.columns:
    s = pd.to_numeric(df[target], errors="coerce").dropna()
    mean_t = s.mean()
    var_t = s.var()
    disp = var_t / max(mean_t, 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.patch.set_facecolor(BG)

    vals, counts = np.unique(s.astype(int), return_counts=True)
    axes[0].bar(vals, counts, color=C1, edgecolor=BG, linewidth=0.4, zorder=3)
    axes[0].axvline(mean_t, color=C2, lw=2, ls="--", label=f"Media {mean_t:.2f}")
    axes[0].axvline(s.median(), color=C3, lw=2, ls=":", label=f"Mediana {s.median():.0f}")
    axes[0].set_title(f"{target} · distribución discreta", pad=12)
    axes[0].set_xlabel("Triples convertidos", color=MUTED)
    axes[0].set_ylabel("Frecuencia", color=MUTED)
    axes[0].legend()

    axes[1].boxplot(
        s, vert=True, patch_artist=True,
        boxprops=dict(facecolor=C2, alpha=0.5),
        medianprops=dict(color=C1, linewidth=2),
        whiskerprops=dict(color=MUTED),
        capprops=dict(color=MUTED),
        flierprops=dict(marker="o", color=C1, alpha=0.3, markersize=3)
    )
    q1, q3 = s.quantile(.25), s.quantile(.75)
    out_n = ((s < q1 - 1.5*(q3-q1)) | (s > q3 + 1.5*(q3-q1))).sum()
    axes[1].set_title(f"Boxplot · {out_n} valores fuera de 1.5×IQR", pad=12)
    axes[1].set_ylabel("Triples convertidos", color=MUTED)

    axes[2].bar(["Media", "Varianza"], [mean_t, var_t],
                color=[C2, C1 if var_t > mean_t else C5], width=0.5, zorder=3)
    axes[2].set_title(f"Dispersión descriptiva · var/media = {disp:.2f}", pad=12)
    axes[2].set_ylabel("Valor", color=MUTED)
    for bar, val in zip(axes[2].patches, [mean_t, var_t]):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f"{val:.3f}", ha="center", color=MUTED, fontsize=9)

    for ax in axes:
        ax.spines["left"].set_color(BG3)
        ax.spines["bottom"].set_color(BG3)

    fig.suptitle(f"Variable dependiente: {target}", fontsize=12, fontweight="bold", color=TEXT, x=0.02, ha="left")
    plt.tight_layout()
    zoom_plot(fig, f"Distribución de {target}")
    plt.show()

    metrics({
        "Media": f"{mean_t:.3f}",
        "Mediana": f"{s.median():.3f}",
        "Varianza": f"{var_t:.3f}",
        "Var/Media": f"{disp:.3f}",
        "Máximo": f"{s.max():.0f}",
    }, C2)


### 2.10 Clustering jerárquico de variables

Las variables explicativas se agrupan según su comportamiento conjunto. Cuando el método es `ward`, se utiliza correctamente distancia euclidiana sobre las variables normalizadas; Ward no debe aplicarse directamente sobre una matriz arbitraria de distancias de correlación.


In [ ]:
# @title
section(
    "Clustering jerárquico de variables",
    f"Linkage {CONFIG['CLUSTER_LINKAGE']} · {CONFIG['CLUSTER_N_GROUPS']} grupos · explicativas normalizadas",
    C3
)

_excl = set(CONFIG["ID_COLS"]) | {CONFIG["TARGET_COL"]}
_excl |= {c for c in df.columns if c.endswith("_ID")}
_excl |= {c for c in df.columns if c.startswith("NEXT_GAME_")}
_excl |= {c for c in df.columns if c.startswith("TARGET_") and c != CONFIG["TARGET_COL"]}

num_feat = [c for c in df_scaled.select_dtypes("number").columns if c not in _excl]
df_num = df_scaled[num_feat].copy()
df_num = df_num.loc[:, df_num.std(skipna=True) > 0]
df_imp = df_num.fillna(df_num.median())

target_series = pd.to_numeric(df[CONFIG["TARGET_COL"]], errors="coerce")

if df_imp.shape[1] < 2:
    raise ValueError("Se requieren al menos dos variables numéricas explicativas no constantes para el clustering.")

linkage_method = CONFIG["CLUSTER_LINKAGE"].lower()
if linkage_method == "ward":
    # Corrección metodológica: Ward requiere observaciones en espacio euclidiano.
    Z = linkage(df_imp.T.values, method="ward", metric="euclidean")
    distance_label = "Distancia euclidiana acumulada (Ward)"
    cluster_basis = "Variables Min–Max como vectores sobre los partidos"
else:
    corr_mat_for_distance = df_imp.corr().abs()
    dist_mat = 1 - corr_mat_for_distance
    np.fill_diagonal(dist_mat.values, 0)
    dist_condensed = squareform(np.maximum(dist_mat.values, 0), checks=False)
    Z = linkage(dist_condensed, method=linkage_method)
    distance_label = "Distancia 1 − |r|"
    cluster_basis = "Distancia por correlación absoluta"

labels_k = fcluster(Z, t=CONFIG["CLUSTER_N_GROUPS"], criterion="maxclust")
cluster_map = pd.DataFrame({"columna": df_imp.columns, "cluster": labels_k})
cluster_map = cluster_map.merge(
    variable_dictionary[["columna", "grupo", "descripcion_breve"]],
    on="columna", how="left"
)
cluster_map["cluster_label"] = cluster_map["cluster"].map(lambda x: f"G{x:02d}")

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

k = CONFIG["CLUSTER_N_GROUPS"]
cut_height = Z[-(k - 1), 2] if 1 < k <= len(Z) else Z[-1, 2]
dendrogram(
    Z, labels=df_imp.columns.tolist(), ax=ax,
    leaf_rotation=90, leaf_font_size=5,
    color_threshold=cut_height, above_threshold_color=MUTED
)
ax.axhline(cut_height, color=C1, ls="--", lw=1.5, label=f"Corte → {k} grupos")
ax.set_title(f"Dendrograma · {linkage_method.title()} · {k} grupos", pad=16)
ax.set_xlabel("Variables", color=MUTED, fontsize=8)
ax.set_ylabel(distance_label, color=MUTED, fontsize=8)
ax.legend(fontsize=8)
ax.spines["left"].set_color(BG3)
ax.spines["bottom"].set_color(BG3)

col_to_grp = cluster_map.set_index("columna")["cluster"].to_dict()
for lbl in ax.get_xmajorticklabels():
    g = col_to_grp.get(lbl.get_text(), 1)
    lbl.set_color(PALETTE[(g - 1) % len(PALETTE)])

plt.tight_layout()
zoom_plot(fig, f"Dendrograma — {k} clusters")
plt.show()

alert(f"Base del clustering: {cluster_basis}.", "info")
grp_size = cluster_map["cluster_label"].value_counts().sort_index()
metrics({g: str(v) for g, v in grp_size.items()}, C3)

# Representante de cada cluster: mayor correlación absoluta con la variable dependiente.
reps = []
for g in sorted(cluster_map["cluster"].unique()):
    cols_g = cluster_map.loc[cluster_map["cluster"] == g, "columna"].tolist()
    if cols_g:
        corrs_g = df_imp[cols_g].corrwith(target_series).abs().dropna()
        reps.append(corrs_g.idxmax() if len(corrs_g) else cols_g[0])

hm_data = df_imp[reps].copy()
hm_data[CONFIG["TARGET_COL"]] = target_series
hm_cols = [CONFIG["TARGET_COL"]] + [c for c in reps if c != CONFIG["TARGET_COL"]]
hm = hm_data[hm_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
im = ax.imshow(hm.values, cmap=CMAP_DIV, vmin=-1, vmax=1, aspect="auto")
cb = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cb.ax.yaxis.set_tick_params(color=MUTED)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=7)
cb.set_label("Pearson r", color=MUTED, fontsize=8)
ax.set_xticks(range(len(hm_cols)))
ax.set_yticks(range(len(hm_cols)))
ax.set_xticklabels([c[:18] for c in hm_cols], rotation=45, ha="right", fontsize=7, color=MUTED)
ax.set_yticklabels([c[:18] for c in hm_cols], fontsize=7, color=MUTED)
for i in range(len(hm_cols)):
    for j in range(len(hm_cols)):
        v = hm.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                color=BG if abs(v) > 0.6 else TEXT,
                fontweight="700" if abs(v) > 0.7 else "normal")
ax.set_title("Correlación entre representantes de clusters y variable dependiente", pad=14)
for spine in ax.spines.values():
    spine.set_color(BG3)
plt.tight_layout()
zoom_plot(fig, "Heatmap de representantes")
plt.show()

display(tbl(
    cluster_map.sort_values(["cluster", "columna"]).style
    .map(lambda v: (f"color:{PALETTE[(int(v[1:])-1)%len(PALETTE)]};font-weight:700"
                    if isinstance(v, str) and v.startswith("G") else ""),
         subset=["cluster_label"])
))


### 2.11 Perfil de grupos de variables


In [ ]:
# @title
section("Perfil de grupos de variables", "Top relaciones con TARGET_FG3M por cluster", C4)

for g in sorted(cluster_map["cluster"].unique()):
    cols_g = cluster_map.loc[cluster_map["cluster"] == g, "columna"].tolist()
    color_g = PALETTE[(g - 1) % len(PALETTE)]
    win_tag = next(
        (w for w in ["L3", "L5", "L10", "L20", "SEASON", "TREND", "PER_MIN", "FG3"]
         if sum(w in c for c in cols_g) > len(cols_g) * 0.3),
        "MIXTO"
    )

    corrs_g = df_imp[cols_g].corrwith(target_series).dropna() if cols_g else pd.Series(dtype=float)
    top3 = corrs_g.abs().sort_values(ascending=False).head(3)
    top3_html = ", ".join([
        f'<code style="color:{C2};">{c}</code> <span style="color:{MUTED};">(|r|={v:.3f})</span>'
        for c, v in top3.items()
    ]) if len(top3) else "—"

    display(HTML(
        f'<div style="border-left:4px solid {color_g};padding:10px 16px;'
        f'margin:6px 0;background:{BG2};border-radius:0 4px 4px 0;font-family:Arial,sans-serif;">'
        f'<div style="display:flex;align-items:baseline;gap:12px;">'
        f'<span style="font-weight:700;color:{color_g};font-size:14px;">G{g:02d}</span>'
        f'<span style="color:{MUTED};font-size:11px;">{len(cols_g)} variables</span>'
        f'<span style="background:{BG3};color:{color_g};font-size:9px;letter-spacing:1px;'
        f'padding:2px 8px;border-radius:10px;text-transform:uppercase;">{win_tag}</span></div>'
        f'<div style="font-size:11px;color:{TEXT};margin:6px 0 4px;">Mayor asociación con {CONFIG["TARGET_COL"]}: {top3_html}</div>'
        f'<div style="font-size:10px;color:{MUTED};">' + " · ".join(cols_g[:10])
        + (" · ..." if len(cols_g) > 10 else "") + '</div></div>'
    ))


### 2.12 Asociación con `TARGET_FG3M`

La correlación resume asociación lineal, no causalidad. Como la normalización Min–Max es una transformación lineal, no modifica el coeficiente de correlación de Pearson.


In [ ]:
# @title
section("Asociación con variable dependiente", f"Top variables relacionadas con {CONFIG['TARGET_COL']}", C1)

corrs_signed = df_imp.corrwith(target_series).dropna().sort_values(key=lambda s: s.abs(), ascending=False)
top30_signed = corrs_signed.head(30)
top30_abs = top30_signed.abs()
grp_c = cluster_map.set_index("columna")["cluster"].to_dict()
bar_cols = [PALETTE[(grp_c.get(c, 1) - 1) % len(PALETTE)] for c in top30_abs.index]

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor(BG)
brs = ax.barh(top30_abs.index[::-1], top30_abs.values[::-1],
              color=bar_cols[::-1], height=0.6, zorder=3)
for bar, val in zip(brs, top30_abs.values[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", color=MUTED, fontsize=7.5)

seen, handles = set(), []
for c, col in zip(bar_cols, top30_abs.index):
    g = grp_c.get(col, 1)
    if g not in seen:
        seen.add(g)
        handles.append(mpatches.Patch(color=PALETTE[(g-1) % len(PALETTE)], label=f"Cluster G{g:02d}"))
ax.legend(handles=handles, loc="lower right", fontsize=7, ncol=2, facecolor=BG2, edgecolor=BG3)
ax.axvline(0.3, color=BG3, lw=1.2, ls="--")
ax.axvline(0.5, color=C1, lw=1, ls="--", alpha=0.4)
ax.set_title(f"Top 30 · |Pearson r| con {CONFIG['TARGET_COL']} · color = cluster", pad=14)
ax.set_xlabel("|Pearson r|", color=MUTED, fontsize=8)
ax.spines["left"].set_color(BG3)
ax.spines["bottom"].set_color(BG3)
plt.tight_layout()
zoom_plot(fig, f"Top relaciones con {CONFIG['TARGET_COL']}")
plt.show()

top20_df = pd.DataFrame({
    "variable": top30_signed.head(20).index,
    "pearson_r": top30_signed.head(20).values,
    "abs_r": top30_signed.head(20).abs().values,
})
top20_df = top20_df.merge(
    cluster_map[["columna", "cluster_label", "grupo", "descripcion_breve"]],
    left_on="variable", right_on="columna", how="left"
)

display(tbl(
    top20_df[["variable", "pearson_r", "abs_r", "cluster_label", "grupo", "descripcion_breve"]]
    .style
    .format({"pearson_r": "{:.4f}", "abs_r": "{:.4f}"})
    .background_gradient(cmap=CMAP_SEQ, subset=["abs_r"])
))


### 2.13 Exploración interactiva: dependiente versus explicativa

Selecciona una variable numérica explicativa en el **combobox**. El eje X usa su versión Min–Max `[0,1]`; el eje Y mantiene `TARGET_FG3M` en cantidad real de triples. La línea representa una tendencia lineal descriptiva.


In [ ]:
# @title
section("Dispersión interactiva", "Variable explicativa normalizada vs TARGET_FG3M", C2)

scatter_options = []
# Ensure corrs_signed is robustly used if available and not empty
if "corrs_signed" in globals() and not corrs_signed.empty:
    scatter_options = corrs_signed.index.tolist()
else:
    # Fallback: get all numerical features that are not excluded, if corrs_signed is not usable.
    # This assumes df_scaled and CONFIG are correctly set up, which they are.
    _excl_cols = set(CONFIG["ID_COLS"]) | {CONFIG["TARGET_COL"]}
    _excl_cols |= {c for c in df.columns if c.endswith("_ID")}
    _excl_cols |= {c for c in df.columns if c.startswith("NEXT_GAME_")}
    _excl_cols |= {c for c in df.columns if c.startswith("TARGET_") and c != CONFIG["TARGET_COL"]}
    scatter_options = [c for c in df_scaled.select_dtypes("number").columns if c not in _excl_cols]

if not scatter_options:
    alert("No hay variables numéricas explicativas disponibles para el gráfico. Esto puede deberse a que no hay columnas numéricas válidas, son identificadores, o no correlacionan con la variable objetivo.", "warning")
else:
    print(f"Variables disponibles para selección ({len(scatter_options)}):")
    print(f"  {', '.join(scatter_options[:10])}{'...' if len(scatter_options) > 10 else ''}")

    scatter_combo = widgets.Combobox(
        options=scatter_options,
        value=scatter_options[0], # Safe due to 'if not scatter_options' guard
        description="Variable X:",
        ensure_option=True,
        continuous_update=False,
        layout=widgets.Layout(width="620px")
    )
    scatter_button = widgets.Button(
        description="Actualizar gráfico",
        button_style="",
        layout=widgets.Layout(width="180px")
    )
    scatter_output = widgets.Output()

    description_lookup = variable_dictionary.set_index("columna")["descripcion_breve"].to_dict()

    def render_scatter(_=None):
        with scatter_output:
            clear_output(wait=True)
            x_col = scatter_combo.value
            if x_col not in df_scaled.columns: # Check against df_scaled for robustness
                print("Selecciona una variable válida de la lista.")
                return

            plot_df = pd.DataFrame({
                "x": pd.to_numeric(df_scaled[x_col], errors="coerce"),
                "y": pd.to_numeric(df[CONFIG["TARGET_COL"]], errors="coerce"),
            }).dropna()

            if len(plot_df) < 3:
                print("No hay suficientes observaciones válidas para graficar.")
                return

            corr_p = plot_df["x"].corr(plot_df["y"], method="pearson")
            corr_s = plot_df["x"].corr(plot_df["y"], method="spearman")

            if len(plot_df) > CONFIG["SCATTER_MAX_POINTS"]:
                plot_sample = plot_df.sample(CONFIG["SCATTER_MAX_POINTS"], random_state=CONFIG["SEED"])
                sample_note = f"Muestra visual de {len(plot_sample):,} sobre {len(plot_df):,} observaciones"
            else:
                plot_sample = plot_df
                sample_note = f"{len(plot_df):,} observaciones"

            fig, ax = plt.subplots(figsize=(11, 6))
            fig.patch.set_facecolor(BG)
            ax.scatter(plot_sample["x"], plot_sample["y"], s=18, alpha=0.28,
                       color=C2, edgecolors="none", zorder=2)

            if plot_df["x"].nunique() > 1:
                slope, intercept = np.polyfit(plot_df["x"], plot_df["y"], 1)
                x_line = np.linspace(plot_df["x"].min(), plot_df["x"].max(), 200)
                y_line = intercept + slope * x_line
                ax.plot(x_line, y_line, color=C1, lw=2.2,
                        label=f"Tendencia: y = {intercept:.2f} + {slope:.2f}x", zorder=3)
                ax.legend()
            else:
                slope = np.nan

            ax.set_title(f"{CONFIG['TARGET_COL']} vs {x_col}", pad=14)
            ax.set_xlabel(f"{x_col} · normalizada [0,1]", color=MUTED)
            ax.set_ylabel(f"{CONFIG['TARGET_COL']} · triples", color=MUTED)
            ax.spines["left"].set_color(BG3)
            ax.spines["bottom"].set_color(BG3)
            plt.tight_layout()
            zoom_plot(fig, f"Dispersión {x_col} vs {CONFIG['TARGET_COL']}")
            plt.show()

            metrics({
                "N": f"{len(plot_df):,}",
                "Pearson r": f"{corr_p:.3f}",
                "Spearman ρ": f"{corr_s:.3f}",
                "Pendiente": "—" if pd.isna(slope) else f"{slope:.3f}",
            }, C2)
            alert(f"{description_lookup.get(x_col, 'Variable explicativa.')} {sample_note}.", "info")

    scatter_button.on_click(render_scatter)
    display(widgets.HBox([scatter_combo, scatter_button]), scatter_output)
    render_scatter()


### 2.14 Evolución por ventana temporal

Las ventanas `L3`, `L5`, `L10`, `L20` y `SEASON` describen distintos horizontes históricos. Las ventanas cortas suelen ser más reactivas y las largas más estables.


In [ ]:
# @title
section("Evolución por ventana temporal", "FG3M en todas las ventanas — season · L20 · L10 · L5 · L3", C2)

_base = CONFIG["TARGET_COL"].replace("TARGET_","").split("_")[0]  # "FG3M"
window_candidates = {}
for w in CONFIG["WINDOWS"]:
    for pat in [f"AVG_{_base}_{w}", f"{_base}_AVG_{w}",
                f"LAST_{w[1:]}_{_base}" if w.startswith("L") else f"{_base}_{w}",
                f"AVG_{_base.replace('M','')}M_{w}"]:
        if pat in df.columns:
            window_candidates[w] = pat
            break

print(f"Ventanas encontradas para {_base}:")
for k, v in window_candidates.items():
    print(f"  {k}: {v}")

if len(window_candidates) >= 2:
    win_cols   = list(window_candidates.values())
    win_labels = list(window_candidates.keys())
    colors_w   = [C1, C3, C2, C4, C5][:len(win_cols)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(BG)

    for col, label, color in zip(win_cols, win_labels, colors_w):
        s = df[col].dropna()
        if len(s):
            s.hist(bins=20, ax=axes[0], color=color, alpha=0.55,
                   edgecolor=BG, linewidth=0.3, zorder=3, label=label, density=True)
    axes[0].set_title(f"Distribución por ventana · {_base}", pad=12)
    axes[0].set_xlabel("Promedio de triples (FG3M)", color=MUTED)
    axes[0].set_ylabel("Densidad", color=MUTED)
    axes[0].legend()
    axes[0].spines["left"].set_color(BG3)
    axes[0].spines["bottom"].set_color(BG3)

    data_bx = [df[c].dropna().values for c in win_cols]
    bp = axes[1].boxplot(data_bx, patch_artist=True, labels=win_labels,
                          medianprops=dict(color=TEXT, linewidth=1.5))
    for patch, color in zip(bp["boxes"], colors_w):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    for el in ["whiskers","caps","fliers"]:
        for item in bp[el]: item.set_color(MUTED)
    axes[1].set_title(f"Variabilidad por ventana · {_base}", pad=12)
    axes[1].set_ylabel("Triples promedio", color=MUTED)
    axes[1].spines["left"].set_color(BG3)
    axes[1].spines["bottom"].set_color(BG3)

    fig.suptitle(f"Ventanas temporales — {_base}  ·  Comparación exploratoria de ventanas",
                 fontsize=12, fontweight="bold", color=TEXT, x=0.02, ha="left")
    plt.tight_layout()
    zoom_plot(fig, f"Ventanas temporales — {_base}")
    plt.show()

    ev_df = pd.DataFrame({w: df[c].describe() for w, c in zip(win_labels, win_cols)})
    ev_df.loc["cv"] = ev_df.loc["std"] / ev_df.loc["mean"].abs()
    display(tbl(ev_df.style.format("{:.3f}").background_gradient(cmap=CMAP_SEQ, axis=1)))
    alert("cv = std/mean. Ventanas cortas (L3, L5) tienen mayor cv → más reactivas, capturan hot hand. "
          "Season es la más estable. La comparación permite observar estabilidad y reactividad entre ventanas.", "info")
else:
    alert(f"No se encontraron columnas de ventana para {_base}. "
          "Revisa los nombres exactos de columna con la metadata.", "warning")


### 2.15 Distribuciones por grupo de cluster


In [ ]:
# @title
section("Distribuciones por grupo de cluster", "Variables representativas normalizadas · histogramas", C5)

n_g = cluster_map["cluster"].nunique()
fig, axes = plt.subplots(n_g, 3, figsize=(16, max(1, n_g) * 3.2), squeeze=False)
fig.patch.set_facecolor(BG)
fig.suptitle(f"Distribución de variables representativas · {n_g} clusters",
             fontsize=13, fontweight="bold", color=TEXT, y=1.01)

for idx, g in enumerate(sorted(cluster_map["cluster"].unique())):
    cols_g = cluster_map.loc[cluster_map["cluster"] == g, "columna"].tolist()
    color_g = PALETTE[(g - 1) % len(PALETTE)]
    corrs = df_imp[cols_g].corrwith(target_series).abs().sort_values(ascending=False) if cols_g else pd.Series(dtype=float)
    top3 = corrs.head(3).index.tolist() if len(corrs) else cols_g[:3]

    for j in range(3):
        ax = axes[idx, j]
        ax.set_facecolor(BG)
        if j < len(top3):
            col = top3[j]
            s = df_imp[col].dropna()
            s.hist(bins=25, ax=ax, color=color_g, edgecolor=BG,
                   linewidth=0.3, alpha=0.85, zorder=3)
            ax.axvline(s.mean(), color=TEXT, lw=1.2, ls="--", alpha=0.7)
            ax.set_title(f"G{g:02d} · {col[:22]}", fontsize=8, color=TEXT, pad=6)
            ax.set_xlabel("Escala Min–Max [0,1]", fontsize=7, color=MUTED)
        else:
            ax.set_visible(False)
            continue
        ax.tick_params(labelsize=6, colors=MUTED)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color(BG3)
        ax.spines["bottom"].set_color(BG3)

plt.tight_layout()
zoom_plot(fig, "Distribuciones por cluster")
plt.show()


---
## Resumen del análisis exploratorio


In [ ]:
# @title
section("Resumen de hallazgos", "EDA de variables NBA y TARGET_FG3M", C1)

target_col = CONFIG["TARGET_COL"]
ts = pd.to_numeric(df[target_col], errors="coerce").dropna() if target_col in df.columns else pd.Series(dtype=float)
n_clusters = cluster_map["cluster"].nunique() if "cluster_map" in globals() else 0
n_warns = (quality_df["status"] != "ok").sum() if "quality_df" in globals() else 0
mean_t = ts.mean() if len(ts) else np.nan
var_t = ts.var() if len(ts) else np.nan
disp = var_t / max(mean_t, 1e-8) if len(ts) else np.nan
n_scaled = (scaling_report["estado"] == "normalizada").sum() if "scaling_report" in globals() else 0

metrics({
    "Dataset": f"{df.shape[0]:,} × {df.shape[1]:,}",
    "Dependiente": target_col,
    "Normalizadas": str(n_scaled),
    "Clusters": str(n_clusters),
    "Alertas": str(n_warns),
    "Var/Media": "—" if pd.isna(disp) else f"{disp:.2f}",
}, C1)

hallazgos = [
    ("Interpretación", f"Se generó un diccionario para {len(variable_dictionary):,} columnas con abreviaturas NBA y descripción breve."),
    ("Estandarización", f"Se creó df_scaled mediante Min–Max [0,1]; df conserva los valores originales."),
    ("Variable dependiente", f"{target_col}: media={mean_t:.2f}, varianza={var_t:.2f}, ratio var/media={disp:.2f}." if len(ts) else "Variable dependiente no disponible."),
    ("Calidad", f"{n_warns} columnas presentan alguna observación de calidad o temporalidad."),
    ("Clustering", f"Se identificaron {n_clusters} grupos de variables explicativas mediante {CONFIG['CLUSTER_LINKAGE']}."),
    ("Relaciones", "Se calcularon correlaciones y se incorporó un combobox para explorar cada explicativa frente a TARGET_FG3M con línea de tendencia."),
    ("Alcance", "El notebook finaliza en el EDA: no entrena modelos y no exporta artefactos."),
]

rows_h = "".join([
    f'<tr>'
    f'<td style="padding:10px 14px;border-bottom:1px solid {BG3};font-size:9px;'
    f'letter-spacing:1.5px;text-transform:uppercase;color:{MUTED};'
    f'white-space:nowrap;font-weight:700;">{title}</td>'
    f'<td style="padding:10px 14px;border-bottom:1px solid {BG3};font-size:12px;'
    f'color:{TEXT};font-family:Arial,sans-serif;">{description}</td>'
    f'</tr>'
    for title, description in hallazgos
])

display(HTML(
    f'<table style="border-collapse:collapse;width:100%;border-top:3px solid {C1};">{rows_h}</table>'
))
